# Taller 2 — Resolución de problemas

**Curso:** Programación Avanzada  
**Sesión:** 18  
**Kernel:** xeus-cling (C++17)

---

Dos problemas nuevos para enviar al juez. Usa este cuaderno para probar localmente antes de enviar.

- **El profesor y los nombres**: buscar el nombre conocido más cercano en una fila y construir la respuesta
- **Once Cifras**: partir 11 dígitos en 3 partes y minimizar la suma — fuerza bruta sobre 45 combinaciones

La idea central: a veces la solución óptima es simplemente probar todas las posibilidades cuando el espacio de búsqueda es pequeño.

---

## Requisitos previos

Ejecuta esta celda primero:

In [ ]:
#include <iostream>
#include <vector>
#include <string>
#include <sstream>
#include <climits>
using namespace std;

---

## Parte 1 — El profesor y los nombres

### 1.1 El problema

Un profesor conoce algunos nombres; el resto son `?`. Para llamar a un alumno cuyo nombre no conoce, usa al alumno conocido más cercano.

```
Posiciones:  1   2   3   4   5   6   7   8   9   10
Nombres:     A   ?   ?   B   ?   ?   ?   C   ?   ?
```

| Posición | Salida |
|----------|--------|
| 3 | `izquierda de B` (B está en pos. 4, distancia 1; más cerca que A en pos. 1, distancia 2) |
| 8 | `C` (nombre conocido) |
| 6 | `justo enmedio de B y C` (B en pos. 4: distancia 2; C en pos. 8: distancia 2) |
| 10 | `derecha derecha de C` (C en pos. 8: distancia 2, único candidato) |

### 1.2 Análisis

Para cada consulta `p`, el algoritmo es:

```
Si nombres[p] es conocido → imprimir el nombre
Si no:
  Buscar el más cercano a la izquierda  → dL, nombreL
  Buscar el más cercano a la derecha   → dR, nombreR

  Si solo hay izquierda:  "derecha" × dL + "de nombreL"
  Si solo hay derecha:    "izquierda" × dR + "de nombreR"
  Si dL < dR:             "derecha" × dL + "de nombreL"
  Si dR < dL:             "izquierda" × dR + "de nombreR"
  Si dL == dR:            "justo enmedio de nombreL y nombreR"
```

> **¿Por qué "izquierda de B" cuando p=3?**  
> B está a la *derecha* de la posición 3. El alumno está a la izquierda de B, entonces se dice "izquierda de B".
> La dirección en la salida es desde el nombre conocido hacia el alumno desconocido.

### 1.3 Implementación

In [ ]:
{
    string entrada = R"(10
A ? ? B ? ? ? C ? ?
4
3
8
6
10
)";
    istringstream in(entrada);

    int n;
    in >> n;
    vector<string> nom(n);
    for (auto& s : nom) in >> s;

    int m;
    in >> m;
    while (m--) {
        int p;
        in >> p;
        p--;  // índice 0

        if (nom[p] != "?") {
            cout << nom[p] << "\n";
            continue;
        }

        int dL = -1, dR = -1;
        string nL, nR;

        for (int i = p - 1; i >= 0; i--)
            if (nom[i] != "?") { dL = p - i; nL = nom[i]; break; }
        for (int i = p + 1; i < n; i++)
            if (nom[i] != "?") { dR = i - p; nR = nom[i]; break; }

        if (dL == -1 || (dR != -1 && dR < dL)) {
            for (int i = 0; i < dR; i++) cout << "izquierda ";
            cout << "de " << nR << "\n";
        } else if (dR == -1 || dL < dR) {
            for (int i = 0; i < dL; i++) cout << "derecha ";
            cout << "de " << nL << "\n";
        } else {
            cout << "justo enmedio de " << nL << " y " << nR << "\n";
        }
    }
    // Salida esperada:
    // izquierda de B
    // C
    // justo enmedio de B y C
    // derecha derecha de C
}

In [ ]:
{
    // Ejemplo 2: 5 alumnos, solo M conocido
    string entrada = "5\nM ? ? ? ?\n2\n4\n5\n";
    istringstream in(entrada);

    int n;
    in >> n;
    vector<string> nom(n);
    for (auto& s : nom) in >> s;

    int m;
    in >> m;
    while (m--) {
        int p;
        in >> p;
        p--;

        if (nom[p] != "?") { cout << nom[p] << "\n"; continue; }

        int dL = -1, dR = -1;
        string nL, nR;
        for (int i = p - 1; i >= 0; i--)
            if (nom[i] != "?") { dL = p - i; nL = nom[i]; break; }
        for (int i = p + 1; i < n; i++)
            if (nom[i] != "?") { dR = i - p; nR = nom[i]; break; }

        if (dL == -1 || (dR != -1 && dR < dL)) {
            for (int i = 0; i < dR; i++) cout << "izquierda ";
            cout << "de " << nR << "\n";
        } else if (dR == -1 || dL < dR) {
            for (int i = 0; i < dL; i++) cout << "derecha ";
            cout << "de " << nL << "\n";
        } else {
            cout << "justo enmedio de " << nL << " y " << nR << "\n";
        }
    }
    // Salida esperada:
    // derecha derecha derecha de M
    // derecha derecha derecha derecha de M
}

### 1.4 Tu solución para enviar al juez

In [ ]:
// Tu solucion aqui

---

## Parte 2 — Once Cifras

### 2.1 El problema

Dado un número de 11 cifras, dividirlo en 3 partes contiguas y encontrar la partición que minimiza la suma de las tres partes.

```
Entrada: 31124510988
Salida:  3750
```

La partición que da 3750 es: `311` + `2451` + `0988` = 311 + 2451 + 988 = 3750.

### 2.2 Análisis

Para partir una cadena de longitud 11 en 3 partes, necesitas 2 puntos de corte `i` y `j` con `1 ≤ i < j ≤ 10`.

```
Parte 1: s[0 .. i-1]    → stoll(s.substr(0, i))
Parte 2: s[i .. j-1]    → stoll(s.substr(i, j-i))
Parte 3: s[j .. 10]     → stoll(s.substr(j))
```

Número de combinaciones: C(10, 2) = **45**. Probarlas todas es trivial.

| i | j | Parte 1 | Parte 2 | Parte 3 | Suma |
|---|---|---------|---------|---------|------|
| 4 | 7 | 3112 | 451 | 988 | 4551 |
| 3 | 7 | 311 | 2451 | 988 | **3750** |
| 4 | 8 | 3112 | 4510 | 988 | 8610 |

> **¿Por qué leer el número como `string` y no como `long long`?**  
> Porque `substr` nos da directamente cada parte como texto, y `stoll` convierte con manejo automático de ceros iniciales. Extraer partes con aritmética (divisiones y módulos de potencias de 10) es más propenso a errores.

### 2.3 Implementación

In [ ]:
{
    string s = "31124510988";
    long long minSuma = LLONG_MAX;
    int n = (int)s.size();

    for (int i = 1; i < n - 1; i++) {
        for (int j = i + 1; j < n; j++) {
            long long a = stoll(s.substr(0, i));
            long long b = stoll(s.substr(i, j - i));
            long long c = stoll(s.substr(j));
            minSuma = min(minSuma, a + b + c);
        }
    }

    cout << minSuma << "\n";
    // Salida esperada: 3750
}

Prueba con un caso extremo: todos los dígitos iguales a 9.

In [ ]:
{
    // Todos 9s: ¿cuál es la partición óptima?
    string s = "99999999999";
    long long minSuma = LLONG_MAX;
    int iOpt = 0, jOpt = 0;
    int n = (int)s.size();

    for (int i = 1; i < n - 1; i++) {
        for (int j = i + 1; j < n; j++) {
            long long a = stoll(s.substr(0, i));
            long long b = stoll(s.substr(i, j - i));
            long long c = stoll(s.substr(j));
            if (a + b + c < minSuma) {
                minSuma = a + b + c;
                iOpt = i; jOpt = j;
            }
        }
    }

    cout << "Suma minima: " << minSuma << "\n";
    cout << "Particion: " << s.substr(0, iOpt)
         << " | " << s.substr(iOpt, jOpt - iOpt)
         << " | " << s.substr(jOpt) << "\n";
}

### 2.4 Tu solución para enviar al juez

In [ ]:
// Tu solucion aqui

---

## Ejercicios

### Ejercicio 1
Para "El profesor y los nombres", construye un caso donde el mismo alumno desconocido tiene tres nombres conocidos equidistantes. ¿Puede ocurrir dado que hay un nombre por posición? Argumenta.

### Ejercicio 2
Para "Once Cifras", ¿cambia la respuesta si el número de entrada tiene un cero interno (e.g., `30000000000`)? Prueba con el código y explica.

### Ejercicio 3
Modifica la solución de "Once Cifras" para que imprima también la partición que logra el mínimo (las tres subcadenas). Prueba con `31124510988`.

In [ ]:
// Tu solucion aqui

---

| Concepto | Técnica | Cuándo usarla |
|----------|---------|---------------|
| Vecino más cercano en lista | Recorrido lineal izq/der, primero que no sea `?` | Listas pequeñas (n ≤ 100) |
| Construir salida variable | Bucle que imprime la palabra `d` veces | Distancia calculada en tiempo de ejecución |
| Partir una cadena en k partes | `substr` + `stoll`, iterar sobre puntos de corte | Cadena de longitud fija, pocas combinaciones |
| Fuerza bruta | Enumerar todas las posibilidades | Espacio de búsqueda ≤ ~10⁶ |

---
*Programación Avanzada — Universidad Panamericana*